# The Dot Product
**Topic:** Linear Algebra for Machine Learning

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Calculate** the dot product of two vectors by multiplying components and summing
- **Interpret** the dot product as a measure of similarity: how much two vectors point in the same direction
- **Explain** why the dot product equals zero when two vectors are perpendicular

> **Tip:** Start by setting both vectors to point in the same direction using the sliders and observe the dot product value. Then rotate one vector 90 degrees until they are perpendicular and watch the dot product reach exactly zero.

---
## How we got here

In *02: Matrix Operations* we added and scaled matrices. Those operations treated each element independently. The dot product is the first operation that connects elements across two vectors, combining each pair of corresponding elements and summing the results into a single number. That single number carries deep geometric meaning and is the engine behind the most important operation in machine learning: the forward pass.

---
## Why this matters for data science

The prediction step in linear regression is a dot product: predicted value = weight vector · feature vector. Cosine similarity in NLP (used to compare word embeddings, document vectors, and recommendation system preferences) is a normalized dot product. Every output neuron in a neural network computes a dot product followed by a non-linear activation. The dot product is the fundamental computation that turns raw numbers into predictions.

---
## Try it yourself

In [2]:
out = Output()

x1_sl = FloatSlider(value=3.0, min=-5.0, max=5.0, step=0.1,
    description="a: x =", style={"description_width": "60px"},
    layout=widgets.Layout(width="360px"))
y1_sl = FloatSlider(value=4.0, min=-5.0, max=5.0, step=0.1,
    description="a: y =", style={"description_width": "60px"},
    layout=widgets.Layout(width="360px"))
x2_sl = FloatSlider(value=1.0, min=-5.0, max=5.0, step=0.1,
    description="b: x =", style={"description_width": "60px"},
    layout=widgets.Layout(width="360px"))
y2_sl = FloatSlider(value=2.0, min=-5.0, max=5.0, step=0.1,
    description="b: y =", style={"description_width": "60px"},
    layout=widgets.Layout(width="360px"))

def render(change=None):
    ax, ay = x1_sl.value, y1_sl.value
    bx, by = x2_sl.value, y2_sl.value

    dot = ax*bx + ay*by
    mag_a = float(np.sqrt(ax**2 + ay**2))
    mag_b = float(np.sqrt(bx**2 + by**2))

    if mag_a > 1e-6 and mag_b > 1e-6:
        cos_val = float(np.clip(dot / (mag_a * mag_b), -1.0, 1.0))
        angle_deg = float(np.degrees(np.arccos(cos_val)))
    else:
        cos_val, angle_deg = 0.0, 90.0

    if dot > 0.01:
        dot_color, dot_label = "#2d9c5f", "Positive — vectors point in similar directions"
    elif dot < -0.01:
        dot_color, dot_label = "#d93025", "Negative — vectors point in opposing directions"
    else:
        dot_color, dot_label = "#888888", "Zero — vectors are perpendicular"

    fig = go.Figure()
    # Vector a
    fig.add_trace(go.Scatter(x=[0, ax], y=[0, ay], mode="lines",
        line=dict(color=PALETTE["primary"], width=3),
        name=f"a = ({ax:.1f}, {ay:.1f})"))
    fig.add_annotation(x=ax, y=ay, ax=0, ay=0,
        xref="x", yref="y", axref="x", ayref="y",
        arrowhead=3, arrowsize=1.5, arrowwidth=2.5,
        arrowcolor=PALETTE["primary"], showarrow=True, text="")
    fig.add_trace(go.Scatter(x=[ax], y=[ay], mode="markers",
        marker=dict(color=PALETTE["primary"], size=9,
                    line=dict(width=2, color="white")),
        showlegend=False))

    # Vector b
    fig.add_trace(go.Scatter(x=[0, bx], y=[0, by], mode="lines",
        line=dict(color=PALETTE["secondary"], width=3),
        name=f"b = ({bx:.1f}, {by:.1f})"))
    fig.add_annotation(x=bx, y=by, ax=0, ay=0,
        xref="x", yref="y", axref="x", ayref="y",
        arrowhead=3, arrowsize=1.5, arrowwidth=2.5,
        arrowcolor=PALETTE["secondary"], showarrow=True, text="")
    fig.add_trace(go.Scatter(x=[bx], y=[by], mode="markers",
        marker=dict(color=PALETTE["secondary"], size=9,
                    line=dict(width=2, color="white")),
        showlegend=False))

    layout = base_layout(
        title=(f"a · b = {dot:.4f}  |  ‖a‖ = {mag_a:.3f}  |  "
               f"‖b‖ = {mag_b:.3f}  |  θ = {angle_deg:.1f}°"),
        xaxis_title="", yaxis_title="")
    layout.update(
        xaxis=dict(range=[-5.5, 5.5], zeroline=True, zerolinecolor="#ccc"),
        yaxis=dict(range=[-5.5, 5.5], zeroline=True, zerolinecolor="#ccc",
                   scaleanchor="x"),
        height=400)
    fig.update_layout(layout)

    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))
        display(HTML(
            f'<div style="font-family:{FONT["family"]}; font-size:14px; '
            f'padding:10px 18px; background:#f8f9fa; border-radius:6px; margin-top:4px;">'
            f'<b style="color:{dot_color}">● {dot_label}</b><br>'
            f'<b>a · b</b> = {ax:.2f}×{bx:.2f} + {ay:.2f}×{by:.2f} = <b>{dot:.4f}</b>'
            f'&emsp;|&emsp;cosine similarity = <b>{cos_val:.4f}</b>'
            f'&emsp;|&emsp;θ = <b>{angle_deg:.1f}°</b>'
            f'</div>'))

for sl in [x1_sl, y1_sl, x2_sl, y2_sl]:
    sl.observe(render, names="value")
display(VBox([HBox([VBox([x1_sl, y1_sl]), VBox([x2_sl, y2_sl])]), out]))
render()

---
## What's happening?

The dot product of two vectors **a** and **b** is computed by multiplying their corresponding components and adding the results:

$$\mathbf{a} \cdot \mathbf{b} = a_1 b_1 + a_2 b_2 + \cdots + a_n b_n = \sum_{i=1}^n a_i b_i$$

For vectors in 2D: if **a** = (3, 4) and **b** = (1, 2), then **a · b** = 3×1 + 4×2 = 3 + 8 = 11.

The dot product can also be expressed using the angle θ between the two vectors:

$$\mathbf{a} \cdot \mathbf{b} = \|\mathbf{a}\| \, \|\mathbf{b}\| \cos(\theta)$$

Here ‖**a**‖ is the **magnitude** (length) of vector **a**, computed as √(a₁² + a₂² + ... + aₙ²).

| Dot product value | What it means geometrically | ML interpretation |
|------------------|-----------------------------|------------------|
| Large positive | Vectors point in similar directions | Feature vector and weight vector are aligned; strong prediction |
| Zero | Vectors are perpendicular | No linear relationship between feature and prediction direction |
| Large negative | Vectors point in opposite directions | Feature vector strongly opposes the weight direction |

### Cosine similarity: the normalized dot product

If you divide the dot product by the magnitudes of both vectors, you get cosine similarity, which equals cos(θ). It ranges from −1 (opposite) to +1 (identical direction), regardless of vector length. This makes it ideal for comparing documents or word embeddings where length should not matter.

$$\text{cosine similarity} = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \, \|\mathbf{b}\|}$$

Return to the widget and verify: set both vectors equal (same direction, any length) and confirm cosine similarity = 1. Set them at 90 degrees and confirm cosine similarity = 0.

---
## Real-world example: Cosine similarity for document comparison in NLP

Word embeddings represent each word as a high-dimensional vector. The similarity between two words or documents is measured by the cosine of the angle between their vectors, a normalized dot product. The chart shows cosine similarity between word pairs, visualized as a heatmap.

Notice:

- **Notice:** Words with similar meanings ("king" and "queen," "cat" and "dog") have high cosine similarity, reflecting that they appear in similar contexts in the training data
- **Notice:** Unrelated words ("king" and "pasta") have low cosine similarity; their embedding vectors point in very different directions in the high-dimensional space
- **Notice:** The diagonal is always 1.0, every word is perfectly similar to itself, which anchors your reading of the off-diagonal values

> **Discussion question:** King − Man + Woman ≈ Queen is the famous word2vec analogy. In terms of dot products and vector directions, why does this arithmetic work? What property of the embedding space makes word analogies expressible as vector addition and subtraction?

In [3]:
# ── Cosine similarity heatmap for word embeddings (simulated) ─────────────────
np.random.seed(42)
words  = ["king", "queen", "man", "woman", "cat", "dog", "pasta", "France"]
dim    = 300   # realistic embedding size; at dim=50 random noise produced
               # spurious cosines near -0.3 for unrelated words, which the
               # diverging colorscale then displayed as meaningful "opposition"

base      = np.random.randn(dim)
royalty_m = base + np.random.randn(dim) * 0.3
royalty_f = base + np.random.randn(dim) * 0.3 + np.random.randn(dim) * 0.5
male      = np.random.randn(dim)
female    = male + np.random.randn(dim) * 0.5
cat_v     = np.random.randn(dim)
dog_v     = cat_v + np.random.randn(dim) * 0.4
pasta_v   = np.random.randn(dim) * 2
france_v  = np.random.randn(dim)

vecs   = np.array([royalty_m, royalty_f, male, female, cat_v, dog_v, pasta_v, france_v])
norms  = np.linalg.norm(vecs, axis=1, keepdims=True)
vecs_n = vecs / norms
cosine_sim = vecs_n @ vecs_n.T

fig = go.Figure(go.Heatmap(
    z=cosine_sim.round(2), x=words, y=words,
    colorscale="RdBu", zmid=0,
    text=cosine_sim.round(2), texttemplate="%{text}",
    colorbar=dict(title="cosine similarity"),
))
layout = base_layout(
    title="Cosine Similarity Between Word Embeddings — Dot Product Divided by Magnitudes",
    xaxis_title="", yaxis_title="",
)
fig.update_layout(layout)
fig.update_yaxes(autorange="reversed")
fig.show()

### Dot product applications in machine learning

| Dot product value | Angle between vectors | Interpretation | ML meaning |
|------------------|-----------------------|---------------|-----------|
| = ‖a‖·‖b‖ (maximum) | 0°: parallel | Perfectly aligned | Strong positive prediction; identical direction |
| > 0 | 0° to 90° | Same general direction | Positive contribution to prediction |
| = 0 | 90°: perpendicular | No relationship | Feature has no effect on this prediction |
| < 0 | 90° to 180° | Opposing directions | Negative contribution to prediction |
| = −‖a‖·‖b‖ (minimum) | 180°: anti-parallel | Perfectly opposed | Strong negative prediction |

---
## Key takeaway

> **The dot product multiplies corresponding components and sums them into a single number measuring how much two vectors align; it is the fundamental computation in linear regression predictions, neural network forward passes, and NLP similarity measures.**

---
*Next up: Matrix Multiplication — extending the dot product to transform entire datasets at once, and why this is the forward pass in a neural network*